## Region extraction by HARs

In [ ]:
import os
import csv
import pathlib
import subprocess
from datetime import datetime
from multiprocessing import Manager
from concurrent.futures import ProcessPoolExecutor, as_completed
import pandas as pd

# Configuration
release = 12

# Make sure bgzip, tabix and plink2 are on PATH
os.environ["PATH"] = "/home/jupyter/tools:" + os.environ.get("PATH", "")

DIR_TOOL = "/home/jupyter/tools"
DIR_WSPS = "/home/jupyter/workspace/ws_files"
DIR_HARS_REF = f"{DIR_WSPS}/HARS_files/HARs_merged"

# The union HAR list must already exist (built by the region merge notebook)
HAR_LIST_FILE = f"{DIR_HARS_REF}/HAR_list_phase_1_union.tsv"

RELEASE_PATH = pathlib.Path(pathlib.Path.home(), f'workspace/gp2_tier2_eu_release{release}')
PATH_NBA_GENO = pathlib.Path(RELEASE_PATH, 'imputed_genotypes')
PATH_WGS_GENO = pathlib.Path(RELEASE_PATH, 'wgs/dragen_joint_calling/plink')

ANCESTRIES = ['AAC', 'AFR', 'AJ', 'AMR', 'CAS', 'EAS', 'EUR', 'FIN', 'MDE', 'SAS', 'CAH']
DATASETS   = ['NBA', 'WGS']

MAX_WORKERS = 16  

# Absolute paths to the binaries (in case PATH isn't propagated to worker processes)
PLINK2 = f"{DIR_TOOL}/plink2"
BGZIP  = f"{DIR_TOOL}/bgzip"
TABIX  = f"{DIR_TOOL}/tabix"


# Load the union HAR list
if not os.path.isfile(HAR_LIST_FILE):
    raise FileNotFoundError(
        f"{HAR_LIST_FILE} not found.\n"
        f"Run the region merge/deduplication notebook first to generate the union list."
    )

with open(HAR_LIST_FILE, 'r') as f:
    reader = csv.reader(f, delimiter='\t')
    HARS_DICT = {
        row[3]: {
            'name':  row[3],
            'chrom': row[0].replace('chr', ''),
            'start': int(row[1]),
            'end':   int(row[2]),
        }
        for row in reader
    }

print(f"Union HAR list loaded from: {HAR_LIST_FILE}")
print(f"Unique regions: {len(HARS_DICT)}")
print(f"Example: {next(iter(HARS_DICT.items()))}")


# Extraction function
def regionExtractor(HAR, chrom, startBP, endBP, sets, PATH_GENO, ANCESTRY,
                    exceptions_file, lock):
    """Extract variants for one HAR in a given ancestry/dataset."""
    # Output goes to the new (_union) directory; covariates are read from the old one
    MAIN_OUTPUT = f"{DIR_WSPS}/results_region_extrac_v3_{sets}/{ANCESTRY}/InputFiles"
    MAIN_COVAR  = f"{DIR_WSPS}/Working_{sets}_v3/{ANCESTRY}/InputFiles"

    # NBA filenames use a _vwb suffix, WGS doesn't - auto-detect which one applies
    candidate_vwb    = f"{PATH_GENO}/{ANCESTRY}/chr{chrom}_{ANCESTRY}_release12_vwb"
    candidate_no_vwb = f"{PATH_GENO}/{ANCESTRY}/chr{chrom}_{ANCESTRY}_release12"
    if os.path.isfile(f"{candidate_vwb}.pgen"):
        inputRawFile = candidate_vwb
    elif os.path.isfile(f"{candidate_no_vwb}.pgen"):
        inputRawFile = candidate_no_vwb
    else:
        with lock:
            with open(exceptions_file, 'a') as f:
                f.write(f'{HAR}\t{ANCESTRY}\t{chrom}\t{startBP}\t{endBP}\tNO_INPUT_PGEN\n')
        return f"[NOINPUT] {HAR} - {ANCESTRY} | neither {candidate_vwb}.pgen nor {candidate_no_vwb}.pgen exists"

    outputDir = f"{MAIN_OUTPUT}/Indiv_HARS"
    covar = f"{MAIN_COVAR}/{ANCESTRY}_covariate_file.txt"
    # Confirmed PD/Control list from the covariate builder notebook
    samplestokeep = f"{MAIN_COVAR}/{ANCESTRY}.samplestokeep"
    outputPrefix = f"{outputDir}/{HAR}"
    os.makedirs(outputDir, exist_ok=True)

    # Resume: skip if a valid vcf.gz + tbi already exists
    if (os.path.isfile(f"{outputPrefix}.vcf.gz") and
        os.path.getsize(f"{outputPrefix}.vcf.gz") > 0 and
        os.path.isfile(f"{outputPrefix}.vcf.gz.tbi")):
        return f"[CACHED] {HAR} - {ANCESTRY}"

    # Check the covariate file exists
    if not os.path.isfile(covar):
        with lock:
            with open(exceptions_file, 'a') as f:
                f.write(f'{HAR}\t{ANCESTRY}\t{chrom}\t{startBP}\t{endBP}\tNO_COVAR\n')
        return f"[NOCOVAR] {HAR} - {ANCESTRY} | {covar} not found"

    # Fail explicitly if samplestokeep is missing
    if not os.path.isfile(samplestokeep):
        with lock:
            with open(exceptions_file, 'a') as f:
                f.write(f'{HAR}\t{ANCESTRY}\t{chrom}\t{startBP}\t{endBP}\tNO_SAMPLESTOKEEP\n')
        return f"[NOSAMPLESTOKEEP] {HAR} - {ANCESTRY} | {samplestokeep} not found"

    # Pass an updated PATH to child worker processes
    env = os.environ.copy()
    env["PATH"] = f"{DIR_TOOL}:" + env.get("PATH", "")

    # plink2 extraction (NBA uses imputation R2, WGS doesn't)
    if sets == "NBA":
        cmd = [
            PLINK2,
            "--pfile", inputRawFile,
            "--chr", str(chrom),
            "--from-bp", str(startBP),
            "--to-bp",   str(endBP),
            "--keep", samplestokeep,  # restrict MAC/MAF calculation to PD/Control only
            "--extract-if-info", "R2>=0.8",
            "--mac", "2",
            "--hwe", "0.0001", "keep-fewhet",
            "--max-maf", "0.05",
            "--make-pgen",
            "--out", outputPrefix
        ]
    else:  # WGS
        cmd = [
            PLINK2,
            "--pfile", inputRawFile,
            "--chr", str(chrom),
            "--from-bp", str(startBP),
            "--to-bp",   str(endBP),
            "--keep", samplestokeep,  # --pheno/--not-pheno don't filter samples, --keep does
            "--mac", "2",
            "--max-maf", "0.05",
            "--make-pgen",
            "--pheno", covar,
            "--not-pheno", "FATID", "MATID", "SEX", "AGE",
                           "PC1","PC2","PC3","PC4","PC5","PC6","PC7","PC8","PC9","PC10",
            "--out", outputPrefix
        ]

    try:
        result = subprocess.run(cmd, capture_output=True, text=True, env=env)
    except Exception as e:
        return f"[ERROR] {HAR} - {ANCESTRY} | plink2 failed to launch: {e}"

    # Expected case: no variants passed filters
    if ("No variants remaining after main filters" in result.stderr or
        "No variants remaining after main filters" in result.stdout):
        with lock:
            with open(exceptions_file, 'a') as f:
                f.write(f'{HAR}\t{ANCESTRY}\t{chrom}\t{startBP}\t{endBP}\tSKIP\n')
        return f"[SKIP]  {HAR} - {ANCESTRY} | no variants passed filters"

    # Other plink2 errors
    if result.returncode != 0:
        with lock:
            with open(exceptions_file, 'a') as f:
                f.write(f'{HAR}\t{ANCESTRY}\t{chrom}\t{startBP}\t{endBP}\tPLINK2_RC{result.returncode}\n')
        return f"[ERROR] {HAR} - {ANCESTRY} | plink2 rc={result.returncode} | {result.stderr[-150:]}"

    # Convert pgen -> vcf -> vcf.gz -> vcf.gz.tbi
    try:
        subprocess.run([
            PLINK2,
            "--pfile", outputPrefix,
            "--recode", "vcf", "id-paste=iid",
            "--out", outputPrefix
        ], capture_output=True, check=True, env=env)
        subprocess.run([BGZIP, "-f", f"{outputPrefix}.vcf"],
                       check=True, capture_output=True, env=env)
        subprocess.run([TABIX, "-f", "-p", "vcf", f"{outputPrefix}.vcf.gz"],
                       check=True, capture_output=True, env=env)
    except subprocess.CalledProcessError as e:
        return f"[ERROR] {HAR} - {ANCESTRY} | VCF conversion failed: rc={e.returncode}"

    return f"[DONE]  {HAR} - {ANCESTRY} | chr{chrom}:{startBP}-{endBP}"


def run_task(args):
    return regionExtractor(*args)


# Main loop
if __name__ == "__main__":
    BUILD = "hg38"

    # Check the binaries exist
    for tool_name, tool_path in [("plink2", PLINK2), ("bgzip", BGZIP), ("tabix", TABIX)]:
        if not os.path.isfile(tool_path):
            raise FileNotFoundError(f"{tool_name} not found at {tool_path}")
        print(f"  {tool_name}: {tool_path}")
    print()

    with Manager() as manager:
        lock = manager.Lock()

        for sets in DATASETS:
            PATH_GENO = str(PATH_NBA_GENO if sets == "NBA" else PATH_WGS_GENO)

            # Pre-create dirs and reset exception files
            for ANCESTRY in ANCESTRIES:
                exc = pathlib.Path(DIR_WSPS, f'results_region_extrac_v3_{sets}', ANCESTRY,
                                   f'failed_HARs_{sets}_{ANCESTRY}.tsv')
                exc.parent.mkdir(parents=True, exist_ok=True)
                with open(exc, 'w') as f:
                    f.write('HAR\tancestry\tchr\tstart\tend\treason\n')
                outDir = pathlib.Path(DIR_WSPS, f'results_region_extrac_v3_{sets}', ANCESTRY,
                                      'InputFiles', 'Indiv_HARS')
                os.makedirs(outDir, exist_ok=True)

            # Build task list
            tasks = [
                (HAR,
                 HARS_DICT[HAR]["chrom"],
                 str(HARS_DICT[HAR]["start"]),
                 str(HARS_DICT[HAR]["end"]),
                 sets,
                 PATH_GENO,
                 ANCESTRY,
                 str(pathlib.Path(DIR_WSPS, f'results_region_extrac_v3_{sets}', ANCESTRY,
                                  f'failed_HARs_{sets}_{ANCESTRY}.tsv')),
                 lock)
                for ANCESTRY in ANCESTRIES
                for HAR in HARS_DICT
            ]

            print(f"\n--- {sets}: {len(tasks)} tasks ({len(HARS_DICT)} HARs x {len(ANCESTRIES)} ancestries) ---")
            print(f"Workers: {MAX_WORKERS}\n")

            ts_start = datetime.now()
            n_done = n_cached = n_skip = n_noinput = n_nocovar = n_err = 0

            with ProcessPoolExecutor(max_workers=MAX_WORKERS) as executor:
                futures = {executor.submit(run_task, task): task for task in tasks}

                for completed, future in enumerate(as_completed(futures), 1):
                    try:
                        result = future.result()
                    except Exception as e:
                        task = futures[future]
                        result = f"[EXC ] {task[0]} - {task[6]} | {e}"

                    if   result.startswith('[DONE'):    n_done    += 1
                    elif result.startswith('[CACHED'):  n_cached  += 1
                    elif result.startswith('[SKIP'):    n_skip    += 1
                    elif result.startswith('[NOINPUT'): n_noinput += 1
                    elif result.startswith('[NOCOVAR'): n_nocovar += 1
                    else:                                n_err     += 1

                    # Print progress every 500 tasks, plus every error
                    if completed % 500 == 0 or result.startswith(('[ERROR', '[EXC ')):
                        elapsed = datetime.now() - ts_start
                        rate = completed / max(elapsed.total_seconds() / 60, 0.01)
                        eta_min = (len(tasks) - completed) / max(rate, 0.01)
                        print(f"[{completed}/{len(tasks)}] "
                              f"D={n_done} C={n_cached} S={n_skip} "
                              f"NI={n_noinput} NC={n_nocovar} E={n_err} "
                              f"| {rate:.0f}/min | ETA {eta_min/60:.1f}h | {result}")

            print(f"\n{sets} finished in {datetime.now() - ts_start}")
            print(f"{sets} summary: DONE={n_done} CACHED={n_cached} SKIP={n_skip} "
                  f"NOINPUT={n_noinput} NOCOVAR={n_nocovar} ERROR={n_err}")

    print("\nAll extraction finished.")

## Summary

In [ ]:
import os, csv
import pandas as pd

DIR_WSPS   = "/home/jupyter/workspace/ws_files"
HAR_LIST_FILE = f"{DIR_WSPS}/HARS_files/HARs_merged/HAR_list_phase_1_union.tsv"
ANCESTRIES = ['AAC','AFR','AJ','AMR','CAS','EAS','EUR','FIN','MDE','SAS','CAH']
DATASETS   = ['NBA','WGS']

with open(HAR_LIST_FILE) as f:
    TOTAL = sum(1 for _ in csv.reader(f, delimiter='\t'))

header = f"{'DS':<5} {'ANC':<5} {'GENERATED':>10} {'SKIP':>6} {'NOINPUT':>8} {'NOCOVAR':>8} {'OTHER':>6} {'%TOT':>7} {'STATUS':<14}"
print(header)
print("-" * len(header))

totals = dict(gen=0, skip=0, ni=0, nc=0, other=0, exp=0)

for ds in DATASETS:
    for anc in ANCESTRIES:
        base     = f"{DIR_WSPS}/results_region_extrac_v3_{ds}"
        vcf_dir  = f"{base}/{anc}/InputFiles/Indiv_HARS"
        fail_tsv = f"{base}/{anc}/failed_HARs_{ds}_{anc}.tsv"

        n_gen = 0
        if os.path.isdir(vcf_dir):
            n_gen = sum(
                1 for f in os.listdir(vcf_dir)
                if f.endswith('.vcf.gz')
                and not f.endswith('.tbi')
                and os.path.getsize(os.path.join(vcf_dir, f)) > 0
                and os.path.isfile(os.path.join(vcf_dir, f) + '.tbi')
            )

        n_skip = n_ni = n_nc = n_other = 0
        if os.path.isfile(fail_tsv):
            try:
                df = pd.read_csv(fail_tsv, sep='\t')
                if 'reason' in df.columns:
                    c = df['reason'].value_counts().to_dict()
                    n_skip  = c.get('SKIP', 0)
                    n_ni    = c.get('NO_INPUT_PGEN', 0)
                    n_nc    = c.get('NO_COVAR', 0)
                    n_other = sum(v for k, v in c.items()
                                  if k not in ('SKIP', 'NO_INPUT_PGEN', 'NO_COVAR'))
            except Exception:
                pass

        acc = n_gen + n_skip + n_ni + n_nc + n_other
        pct = acc / TOTAL * 100 if TOTAL else 0
        if acc == TOTAL:
            status = "complete"
        elif acc >= TOTAL * 0.95:
            status = "almost complete"
        elif acc == 0:
            status = "not processed"
        else:
            status = "partial"

        print(f"{ds:<5} {anc:<5} {n_gen:>10} {n_skip:>6} {n_ni:>8} {n_nc:>8} {n_other:>6} {pct:>6.1f}% {status}")
        totals['gen']   += n_gen
        totals['skip']  += n_skip
        totals['ni']    += n_ni
        totals['nc']    += n_nc
        totals['other'] += n_other
        totals['exp']   += TOTAL

print("-" * len(header))
acc_tot = totals['gen'] + totals['skip'] + totals['ni'] + totals['nc'] + totals['other']
pct_tot = acc_tot / totals['exp'] * 100 if totals['exp'] else 0
print(f"{'TOTAL':<11} {totals['gen']:>10} {totals['skip']:>6} {totals['ni']:>8} {totals['nc']:>8} {totals['other']:>6} {pct_tot:>6.1f}%")

print()
if acc_tot == totals['exp']:
    print("Extraction complete.")
else:
    print(f"Pending: {totals['exp'] - acc_tot} tasks")